In [1]:
import pandas as pd
import re

def ground_truths_escenario3(log_text):
    log = str(log_text).upper()
    
    if 'TYPE=SYSCALL' in log and ('AUID="ROOT"' in log or 'UID="ROOT"' in log) and any(cmd in log for cmd in ['COMM="APT-GET"', 'COMM="SYSTEMCTL"']):
        return ['INFO', 'BAJO', 'INFORMACIÓN']
    elif 'TYPE=SYSCALL' in log and 'COMM="USERADD"' in log:
        return ['MEDIO']
    elif 'ERROR_PARSE' in log or 'FALLO_LLM' in log:
        return ['ERROR']
    else:
        return ['INFO', 'BAJO', 'INFORMACIÓN']

def evaluar_por_orden(path_completo, path_evaluar, funcion_heuristica):
    """
    Evalúa la precisión comparando fila por fila basándose en el orden estricto de los CSV.
    """
    df_completo = pd.read_csv(path_completo)
    df_evaluar = pd.read_csv(path_evaluar)
    
    if len(df_completo) != len(df_evaluar):
        print(f"¡ALERTA!: Archivos desfasados. Completo: {len(df_completo)} | A evaluar: {len(df_evaluar)}")
        return
        
    verdad_en_orden = df_completo['Log'].apply(funcion_heuristica).tolist()
    
    # Contadores para el LLM (Lo que si analizó)
    aciertos_llm = 0
    total_llm_evaluado = 0
    
    # Contadores Globales (LLM + Omisiones)
    aciertos_globales = 0
    aciertos_por_omision = 0
    total_filas = len(df_evaluar)
    
    for i in range(total_filas):
        etiquetas_reales = verdad_en_orden[i]
        
        # Leer la justificación para saber si fue omitido
        justificacion = str(df_evaluar.loc[i, 'Justificacion'] if 'Justificacion' in df_evaluar.columns else '').strip()
        es_omitido = 'Omitido' in justificacion
        
        if es_omitido:
            # Si se omitió se marcó como INFO
            aciertos_globales += 1
            aciertos_por_omision += 1
        else:
            # Si NO se omitió, lo evaluó el LLM
            prediccion_llm = str(df_evaluar.loc[i, 'Riesgo']).upper().strip()
            total_llm_evaluado += 1
            
            if prediccion_llm in etiquetas_reales:
                aciertos_llm += 1
                aciertos_globales += 1
                
    # --- CÁLCULOS ---
    precision_llm = (aciertos_llm / total_llm_evaluado) * 100 if total_llm_evaluado > 0 else 0
    precision_global = (aciertos_globales / total_filas) * 100 if total_filas > 0 else 0
    
    # --- REPORTE ---
    print("="*65)
    print(f"EVALUACIÓN DE: {path_evaluar}")
    print("="*65)
    print(f"Total de eventos en el log     : {total_filas}")
    print(f"Eventos analizados por el LLM  : {total_llm_evaluado}")
    print(f"Eventos omitidos               : {total_filas - total_llm_evaluado}")
    print("-" * 65)
    print(f"1️⃣  PRECISIÓN DEL LLM (Sin omitidos)      : {precision_llm:.2f}% ({aciertos_llm}/{total_llm_evaluado})")
    print(f"2️⃣  PRECISIÓN GLOBAL (Con omitidos)       : {precision_global:.2f}% ({aciertos_globales}/{total_filas})")
    print("="*65)
    
    return precision_llm, precision_global


In [2]:
# --- EJECUCIÓN ---
# Definir la ruta del archivo que tiene la verdad (RAW Completo)
ruta_verdad = '../results/prompt1/escenario3_resultados_raw_completo_phi3mini.csv'

# Evaluar el RAW Completo contra sí mismo para la nota base
evaluar_por_orden(ruta_verdad, ruta_verdad, ground_truths_escenario3)

# Evaluar los otros formatos
evaluar_por_orden(ruta_verdad, '../results/prompt1/escenario3_resultados_raw_reducido_phi3mini.csv', ground_truths_escenario3)
evaluar_por_orden(ruta_verdad, '../results/prompt1/escenario3_resultados_json_reducido_phi3mini.csv', ground_truths_escenario3)

EVALUACIÓN DE: ../results/prompt1/escenario3_resultados_raw_completo_phi3mini.csv
Total de eventos en el log     : 10
Eventos analizados por el LLM  : 10
Eventos omitidos               : 0
-----------------------------------------------------------------
1️⃣  PRECISIÓN DEL LLM (Sin omitidos)      : 10.00% (1/10)
2️⃣  PRECISIÓN GLOBAL (Con omitidos)       : 10.00% (1/10)
EVALUACIÓN DE: ../results/prompt1/escenario3_resultados_raw_reducido_phi3mini.csv
Total de eventos en el log     : 10
Eventos analizados por el LLM  : 10
Eventos omitidos               : 0
-----------------------------------------------------------------
1️⃣  PRECISIÓN DEL LLM (Sin omitidos)      : 70.00% (7/10)
2️⃣  PRECISIÓN GLOBAL (Con omitidos)       : 70.00% (7/10)
EVALUACIÓN DE: ../results/prompt1/escenario3_resultados_json_reducido_phi3mini.csv
Total de eventos en el log     : 10
Eventos analizados por el LLM  : 10
Eventos omitidos               : 0
----------------------------------------------------------------

(10.0, 10.0)